In [18]:
import os
import pandas as pd
import gspread
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import datetime  # Imported module to fix the datetime.datetime error
import time

def main():
    path_realisasi = r"D:\Renstra\Full Month.xlsx"
    path_rkap = r"D:\Renstra\RKAP 25.xlsx"
    path_export = r"D:\Renstra\Rekapitulasi Flatfile sd 2026.xlsx"
    
    df_DN = extract_excel(path_realisasi, "2026", 0)
    print(df_DN.info())
    
    df_target = extract_excel(path_rkap, "Sheet1", 0)
    df_target = df_target[df_target["TAHUN"] == 2026]
    print(df_target.info())
    
    print("####### # Export")
    df_export = extract_excel(path_export, "2026", 0)
    print(df_export.info())
    df_export = df_export[df_export["AREA UPDATE"] == "Export"]
    
    df_dn_export = allDN_Export(df_DN, df_export)
    
    
    if df_dn_export is not None and not df_dn_export.empty:
        print("Data extracted successfully.")
        creds = connect_to_sheet()
        update_sheet('1RvR3fgwWDght4mlhorQxBRnAP4yFUpIoyz73maC1e-o', 'Data Level 1', None, creds, df_dn_export)
    
    df_OA = getOA(df_DN)
    
    if df_OA is not None and not df_OA.empty:
        print("Data extracted successfully.")
        creds = connect_to_sheet()
        update_sheet('1RvR3fgwWDght4mlhorQxBRnAP4yFUpIoyz73maC1e-o', 'Data OA', None, creds, df_OA)
        
    df_Marketing = getMarketing(df_DN)
    
    if df_Marketing is not None and not df_Marketing.empty:
        print("Data extracted successfully.")
        creds = connect_to_sheet()
        update_sheet('1RvR3fgwWDght4mlhorQxBRnAP4yFUpIoyz73maC1e-o', 'Data Marketing', None, creds, df_Marketing)
    
    
    
def allDN_Export(df_DN, df_export):
    # 1. Rename columns in df_DN
    df_DN = df_DN.rename(columns={
        'quantity': 'QTY', 
        'opco_area': 'Entity',
        'bulan': 'Month',
        'tahun': 'Year',
        'revenue': 'Rev'
    })
    
    df_DN['Market'] = 'Dalam Negeri'
    df_export['Market'] = 'Export'
    columns_to_keep = ['QTY', 'Entity', 'Month', 'Year', 'Rev', 'Market']
    
    df_combined = pd.concat([df_DN[columns_to_keep], df_export[columns_to_keep]], ignore_index=True)
    
    # 5. Group and aggregate (include 'Market' in groupby so it stays in the final output)
    df_grouped = df_combined.groupby(['Entity', 'Month', 'Year', 'Market'], as_index=False).agg({
        'Rev': 'sum',
        'QTY': 'sum'
    })
    
    print(df_grouped)
    return df_grouped

def getDN(df_DN):
    print("b")
        
def getOA(df_DN):
    df_DN = df_DN.rename(columns={
        'quantity': 'QTY', 
        'opco_area': 'Entity',
        'bulan': 'Month',
        'tahun': 'Year',
        'revenue': 'Rev',
    })
    columns_to_keep = ['QTY', 'Entity', 'Month', 'Year', 'Rev', 'district_desc', 'province_desc', 'segment_material']
    df_combined = df_DN[columns_to_keep]
    df_grouped = df_combined.groupby(['Entity', 'Month', 'Year', 'district_desc', 'province_desc', 'segment_material'], as_index=False).agg({
        'Rev': 'sum',
        'QTY': 'sum'
    })
    df_grouped['OA_real'] = df_grouped['Rev'] * 0.12
    df_grouped['OA_tar'] = df_grouped['Rev'] * 0.122
    return df_grouped
    
def getMarketing(df_DN):
    df_DN = df_DN.rename(columns={
        'quantity': 'QTY', 
        'opco_area': 'Entity',
        'bulan': 'Month',
        'tahun': 'Year',
        'revenue': 'Rev',
    })
    columns_to_keep = ['QTY', 'Entity', 'Month', 'Year', 'Rev', 'district_desc', 'province_desc']
    df_combined = df_DN[columns_to_keep]
    df_grouped = df_combined.groupby(['Entity', 'Month', 'Year', 'district_desc', 'province_desc'], as_index=False).agg({
        'Rev': 'sum',
        'QTY': 'sum'
    })
    df_grouped['Marketing_Expense_real'] = df_grouped['Rev'] * 0.012
    df_grouped['Marketing_Expense_tar'] = df_grouped['Rev'] * 0.0122
    return df_grouped
    
def setting_header():
    
    print("a")
    
def extract_excel(path, sheet, chosen_header):
    # Added engine='calamine'
    df_result = pd.read_excel(path, sheet_name=sheet, header=chosen_header, engine='calamine')
    print(df_result.head(5))
    return df_result

def create_grouped_dfs(df):
    df_area = df.groupby(['AREA AP', 'province_desc'], as_index=False).agg({
        'revenue_column_name': 'sum',   
        'jml_spj': 'sum'               
    })
    
    # 2. Group by District (include Province)
    # Replace 'district_column_name' with your actual district column name
    df_province = df.groupby(['province_desc', 'district_column_name'], as_index=False).agg({
        'revenue_column_name': 'sum',   # <-- Replace with your actual revenue column name
        'jml_spj': 'sum'                
    })
    return df_area, df_province

def connect_to_sheet():
    SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
    creds = None
    # The file token.json stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first
    # time.
    if os.path.exists(r'C:\Users\mochamad.ilmawan\token.json'):
        creds = Credentials.from_authorized_user_file(r'C:\Users\mochamad.ilmawan\token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'D:\OneDrive\Documents\Direct Selling\MONITORING\Monitoring 2023\client_secret_738271294618-6g8e0hle4jpq8nau1d7b3q9jfsgp0ruk.apps.googleusercontent.com.json', SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open(r'C:\Users\mochamad.ilmawan\token.json', 'w') as token:
            token.write(creds.to_json())
    return creds

def update_sheet(spreadsheetId, spread_range, scopes, creds, df):
    df_clean = df.fillna('')
    header = df_clean.columns.tolist()
    data = df_clean.values.tolist()
    values_with_header = [header] + data
    time.sleep(2)
    try:
        # FIXED: Changed service.sheets() to service.spreadsheets()
        service = build('sheets', 'v4', credentials=creds)
        service.spreadsheets().values().clear(spreadsheetId=spreadsheetId, range=spread_range, body={}).execute()
        
        body = {'values': values_with_header}
        result = service.spreadsheets().values().update(
            spreadsheetId=spreadsheetId, range=spread_range, valueInputOption="USER_ENTERED", body=body).execute()
        print(f"{result.get('updatedCells')} cells updated (sheet replaced).")
        return result
    except HttpError as error:
        print(f"An error occurred: {error}")
        return error
    
    
if __name__ == '__main__':
    main()
    


     SEGMEN AREA AP  tahun  bulan    tanggal  regional regional_update  \
0  CORSALES  AREA 1   2026      1 2026-01-01  CORSALES        CORSALES   
1  CORSALES  AREA 1   2026      1 2026-01-01  CORSALES        CORSALES   
2  CORSALES  AREA 1   2026      1 2026-01-01  CORSALES        CORSALES   
3  CORSALES  AREA 1   2026      1 2026-01-01  CORSALES        CORSALES   
4  CORSALES  AREA 1   2026      1 2026-01-01  CORSALES        CORSALES   

  opco_area  province_code   province_desc  ...  order_reason sales_org  \
0  CORSALES           1010            ACEH  ...           NaN      7900   
1  CORSALES           1011  SUMATERA UTARA  ...           NaN      7900   
2  CORSALES           1011  SUMATERA UTARA  ...           NaN      7900   
3  CORSALES           1011  SUMATERA UTARA  ...           NaN      7900   
4  CORSALES           1011  SUMATERA UTARA  ...           NaN      7900   

  opco_pemilik_brand opco_produsen jml_spj sched_dlvr_date request_dlvr_date  \
0               3000    

      Entity  Month  Year        Market           Rev         QTY
0   CORSALES      1  2026  Dalam Negeri  4.726299e+11  608386.155
1   CORSALES      2  2026  Dalam Negeri  4.615551e+11  590689.830
2   CORSALES      3  2026  Dalam Negeri  3.204361e+11  408126.940
3   CORSALES      4  2026  Dalam Negeri  5.761421e+11  739850.768
4   CORSALES      5  2026  Dalam Negeri  5.788685e+11  748654.020
..       ...    ...   ...           ...           ...         ...
56      TLCC      1  2026        Export  5.451778e+10  105430.000
57      TLCC      2  2026        Export  4.103281e+10   80500.000
58      TLCC      3  2026        Export  1.637839e+10   31500.000
59      TLCC      4  2026        Export  8.254022e+09   16000.000
60      TLCC      5  2026        Export  1.693173e+10   31500.000

[61 rows x 6 columns]
Data extracted successfully.
372 cells updated (sheet replaced).
Data extracted successfully.
37650 cells updated (sheet replaced).
Data extracted successfully.
33741 cells updated (she

In [16]:
convert_xlsx_to_csv(excel_path, csv_path, sheet_name):
    print("Reading Excel file... (this may take a few seconds)")
    df = pd.read_excel(excel_path, sheet_name=sheet_name, engine='calamine')
    df.to_csv(csv_path, index=False)
    print(f"Success! Saved to: {csv_path}")   

SyntaxError: invalid syntax (2335204219.py, line 1)